# Looking at Cyclisation Dataset V1: 

In [7]:
import pandas as pd
import os

# Load the data:
data = pd.read_csv("./df_cyclization_1506_first_uspto_mol.csv", index_col=0)
data.head(5)

,SMILES,MAPPED SMILES,index_atoms,cycled_product,mapped_cycled_product,reaction,mapped_reaction
0,CCOC(=O)[C@@H]1CCc2cc(OS(=O)(=O)C(F)(F)F)cc(=O...,[CH3:1][CH2:2][O:3][C:4](=[O:5])[C@@H:6]1[CH2:...,"(0, 7)",O=C1OCCC2C[C@@H]1n1c2cc(OS(=O)(=O)C(F)(F)F)cc1...,[CH2:1]1[CH2:2][O:3][C:4](=[O:5])[C@@H:6]2[CH2...,CCOC(=O)[C@@H]1CCc2cc(OS(=O)(=O)C(F)(F)F)cc(=O...,[CH3:1][CH2:2][O:3][C:4](=[O:5])[C@@H:6]1[CH2:...
1,CCOC(=O)[C@@H]1CCc2cc(OS(=O)(=O)C(F)(F)F)cc(=O...,[CH3:1][CH2:2][O:3][C:4](=[O:5])[C@@H:6]1[CH2:...,"(1, 5)",CC1OC(=O)[C@@]12CCc1cc(OS(=O)(=O)C(F)(F)F)cc(=...,[CH3:1][CH:2]1[O:3][C:4](=[O:5])[C@@:6]12[CH2:...,CCOC(=O)[C@@H]1CCc2cc(OS(=O)(=O)C(F)(F)F)cc(=O...,[CH3:1][CH2:2][O:3][C:4](=[O:5])[C@@H:6]1[CH2:...
2,CCOC(=O)[C@@H]1CCc2cc(OS(=O)(=O)C(F)(F)F)cc(=O...,[CH3:1][CH2:2][O:3][C:4](=[O:5])[C@@H:6]1[CH2:...,"(5, 7)",CCOC(=O)[C@@]12CC1c1cc(OS(=O)(=O)C(F)(F)F)cc(=...,[CH3:1][CH2:2][O:3][C:4](=[O:5])[C@@:6]12[CH2:...,CCOC(=O)[C@@H]1CCc2cc(OS(=O)(=O)C(F)(F)F)cc(=O...,[CH3:1][CH2:2][O:3][C:4](=[O:5])[C@@H:6]1[CH2:...
3,CCOC(=O)[C@@H]1CCc2cc(OS(=O)(=O)C(F)(F)F)cc(=O...,[CH3:1][CH2:2][O:3][C:4](=[O:5])[C@@H:6]1[CH2:...,"(0, 6)",O=C1OCCC2Cc3cc(OS(=O)(=O)C(F)(F)F)cc(=O)n3[C@H...,[CH2:1]1[CH2:2][O:3][C:4](=[O:5])[C@@H:6]2[CH:...,CCOC(=O)[C@@H]1CCc2cc(OS(=O)(=O)C(F)(F)F)cc(=O...,[CH3:1][CH2:2][O:3][C:4](=[O:5])[C@@H:6]1[CH2:...
4,CCOC(=O)[C@@H]1CCc2cc(OS(=O)(=O)C(F)(F)F)cc(=O...,[CH3:1][CH2:2][O:3][C:4](=[O:5])[C@@H:6]1[CH2:...,"(1, 7)",CC1OC(=O)[C@@H]2CC1c1cc(OS(=O)(=O)C(F)(F)F)cc(...,[CH3:1][CH:2]1[O:3][C:4](=[O:5])[C@@H:6]2[CH2:...,CCOC(=O)[C@@H]1CCc2cc(OS(=O)(=O)C(F)(F)F)cc(=O...,[CH3:1][CH2:2][O:3][C:4](=[O:5])[C@@H:6]1[CH2:...


## Dropping NA values: 

- Only if some are present :)

In [8]:
print("NA values in the data:")
print(data.isna().sum(), "\n")
data = data.dropna()
print(f"We have {data['mapped_reaction'].unique().shape[0]} mapped reactions.")
print(f"We have {data['reaction'].unique().shape[0]} unique reactions.")

NA values in the data:
SMILES                   0
MAPPED SMILES            0
index_atoms              0
cycled_product           0
mapped_cycled_product    0
reaction                 0
mapped_reaction          0
dtype: int64 

We have 39035 mapped reactions.
We have 27329 unique reactions.


## Only keep useful columns: 

In [9]:
columns_to_keep = ["mapped_reaction", "reaction"]
data = data[columns_to_keep]

## Removing RXNs with explicit Hydrogens: 

This is very basic and very prone to bugs: I literally only keep the first product :(

- Fist we check that all the reactions contain one .

In [10]:
len_set = set()
for sample in data["mapped_reaction"]:
    len_set.add(len(sample.split(".")))


data_no_explicit_h = data.copy()


# Remove the explicit hydrogens from the rxn:
data_no_explicit_h["mapped_reaction"] = data_no_explicit_h["mapped_reaction"].apply(
    lambda x: x.split(".")[0]
)
data_no_explicit_h["reaction"] = data_no_explicit_h["reaction"].apply(
    lambda x: x.split(".")[0]
)


# Delete rows with [H][H] still in rxn:
data_no_explicit_h = data_no_explicit_h[
    ~data_no_explicit_h["mapped_reaction"].str.contains(r"\[H\]\[H\]")
]
data_no_explicit_h = data_no_explicit_h[
    ~data_no_explicit_h["reaction"].str.contains(r"\[H\]\[H\]")
]

data_no_explicit_h.mapped_reaction.iloc[-7]

'[CH3:1][O:2][c:3]1[cH:4][cH:5][cH:6][c:7]([CH:8]([NH:9][CH2:10][CH2:11][NH:12][C:13](=[O:14])[O:15][C:16]([CH3:17])([CH3:18])[CH3:19])[C:20](=[O:21])[OH:22])[cH:23]1>>[CH3:1][O:2][c:3]1[cH:4][cH:5][cH:6][c:7]([CH:8]([NH:9][CH2:10][CH2:11][NH:12][C:13](=[O:14])[O:15][C:16]2([CH3:18])[CH2:17][CH2:19]2)[C:20](=[O:21])[OH:22])[cH:23]1'

## Extra Filtering: 

- Only keep 1 of the unique reactions and its mapping. 
- Rename the columns so the data can be used in downstream scripts. 



#### Save the cleaned data with Explicit hydrogen product: 

In [11]:
# cleaned_data = data.drop_duplicates(subset=["reaction"], keep="first")[
#     ["reaction", "mapped_reaction"]
# ]

# cleaned_data = cleaned_data.rename(
#     columns={"reaction": "rxn", "mapped_reaction": "rxn_atom_mapped"}
# )

# cleaned_data.reset_index(drop=True, inplace=True)

# # Save files
# save_directory = "./clean_with_h"
# os.makedirs(save_directory, exist_ok=True)
# cleaned_data.to_csv(
#     os.path.join(save_directory, "cleaned_data.csv"),
#     index=False,
# )
# # cleaned_data.to_json(
# #     os.path.join(save_directory, "cleaned_data.json"),
# # )

#### Save the cleaned data without Explicit hydrogen product

In [12]:
cleaned_data_no_h = data_no_explicit_h.drop_duplicates(
    subset=["reaction"], keep="first"
)[["reaction", "mapped_reaction"]]

cleaned_data_no_h = cleaned_data_no_h.rename(
    columns={"reaction": "rxn", "mapped_reaction": "rxn_atom_mapped"}
)


cleaned_data_no_h.reset_index(drop=True, inplace=True)

# Save files
save_directory = "./clean_no_h"
os.makedirs(save_directory, exist_ok=True)
cleaned_data_no_h.to_csv(
    os.path.join(save_directory, "cleaned_data.csv"),
    index=False,
)
# cleaned_data_no_h.to_json(
#     os.path.join(save_directory, "cleaned_data.json"),
# )